This is used to plot the EV charging stations on map

Fan Yang, 1/11/2025

In [1]:
# initialize and load EV charging station data
# data from: US Department of Energy, Energy efficiency & renewable energy - Alternative Fuels Data Center
# https://afdc.energy.gov/stations#/analyze?country=US&fuel=ELEC&ev_levels=all&access=public&access=private&region=US-CA

import pandas as pd
import folium
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm  # Import tqdm for the progress bar


# Load the CSV file
folder = "/Users/fanyang/Documents/Family/kids/ScienceFair2024/calculation/Geology/"
file_name = "alt_fuel_stations_Jan_12_2025_bayarea.csv" 
EVstation_data = pd.read_csv(folder + file_name, low_memory=False)

# Ensure the required columns exist
required_columns = ["Street Address", "EV Level2 EVSE Num", "Latitude", "Longitude"]
for col in required_columns:
    if col not in EVstation_data.columns:
        raise ValueError(f"CSV file must contain a '{col}' column.")


In [2]:

# Slice the DataFrame to only include the first 100 rows (optional)
# EVstation_data = EVstation_data.head(100)

# Drop rows with missing latitude or longitude values
EVstation_data.dropna(subset=["Latitude", "Longitude"], inplace=True)

# Create a map centered on California
california_map = folium.Map(location=[36.7783, -119.4179], zoom_start=6)

# Add markers for each EV charging station with a progress bar
for index, row in tqdm(EVstation_data.iterrows(), total=len(EVstation_data), desc="Adding Markers"):
    lat, lon = row["Latitude"], row["Longitude"]
    folium.Marker(
        location=[lat, lon],
        popup=f"Address: {row['Street Address']}<br>Level 2 EVSE: {row['EV Level2 EVSE Num']}",
        tooltip=f"Level 2 EVSE: {row['EV Level2 EVSE Num']}",
    ).add_to(california_map)

# Save the map to an HTML file
map_file = folder + "ev_charging_map.html"
california_map.save(map_file)
print(f"Map saved to {map_file}")

# Display the map in a Jupyter Notebook (if applicable)
# california_map

Adding Markers: 100%|███████████████████████| 100/100 [00:00<00:00, 9037.70it/s]

Map saved to ev_charging_map.html


In [ ]:
# Obsolete

# Slice the DataFrame to only include the first 100 rows
EVstation_data = EVstation_data.head(100)

# Initialize the geocoder
geolocator = Nominatim(user_agent="ev_charging_map")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)  # Add rate limiting to avoid API overuse

# Function to geocode addresses
def get_lat_lon(address):
    try:
        location = geocode(address + ", California, USA")
        if location:
            return (location.latitude, location.longitude)
        else:
            return None
    except Exception as e:
        print(f"Error geocoding {address}: {e}")
        return None

# Add latitude and longitude columns to the DataFrame with a progress bar
tqdm.pandas(desc="Geocoding Addresses")  # Initialize tqdm for pandas
EVstation_data["Coordinates"] = EVstation_data["Street Address"].apply(get_lat_lon)

# Drop rows with invalid coordinates
EVstation_data.dropna(subset=["Coordinates"], inplace=True)

# Create a map centered on California
california_map = folium.Map(location=[36.7783, -119.4179], zoom_start=6)

# Add markers for each EV charging station
for index, row in tqdm(EVstation_data.iterrows(), total=len(EVstation_data), desc="Adding Markers"):
    lat, lon = row["Coordinates"]
    folium.Marker(
        location=[lat, lon],
        popup=f"Address: {row['Street Address']}<br>Level 2 EVSE: {row['EV Level2 EVSE Num']}",
        tooltip=f"Level 2 EVSE: {row['EV Level2 EVSE Num']}",
    ).add_to(california_map)

# Save the map to an HTML file
map_file = "ev_charging_map.html"
california_map.save(map_file)
print(f"Map saved to {map_file}")

# Display the map in a Jupyter Notebook (if applicable)
california_map